__Two-Layer MLP Simulations (MLP2)__

Alfred

A 2 --> 2 --> 1 network. The hidden layer is two STAR perceptrons, as in MLP1. The output layer
differs: the two hidden transcripts bind *each other* to form the output complex, and the
constitutive cassette produces the layer-2 *sequestering* RNA instead of a target.

In [ ]:
# Libraries for mathematical analysis
import numpy as np
from skimage import measure
import sympy as sym
import pylab as pl
import matplotlib.pyplot as plt
import scipy.integrate
import biocircuits

# Libraries to visualize results
import bokeh.io
from bokeh.io import show
from bokeh.plotting import figure
from bokeh.plotting import column
import bokeh.palettes
from bokeh.models import LinearColorMapper, ColorBar
from bokeh.models import Range1d
from bokeh.io import export_svgs
from mpl_toolkits.mplot3d import Axes3D
import plotly.graph_objects as go
from bokeh.layouts import row
import re

bokeh.io.output_notebook()

<u>MLP Model</u>

![MLP2 Reaction Scheme](MLP2_Reaction.png)

A two-layer network. Layer 1 holds two STAR perceptrons, $j$ and $k$, identical to MLP1. Each
input plasmid $X_i$ drives **both** a STAR cassette ($\alpha^{+}$) and a sequestering-RNA cassette
($\alpha^{-}$) at each hidden node, so the effective weight
$w_{n,i} \propto \alpha^{+}_{n,i} - \alpha^{-}_{n,i}$ can take either sign.

__How layer 2 differs from MLP1.__ This is the whole point of the architecture, so it is worth
stating precisely:

| | MLP1 | MLP2 (this notebook) |
|---|---|---|
| Output complex $C^{(2)}$ | $Y^{(1)}_j \cdot P^{(2)}$ | $Y^{(1)}_j \cdot Y^{(1)}_k$ |
| What $\theta$ makes | the target $P^{(2)}$ | the sequesterer $Z^{(2)}$ |
| Role of $\theta$ | gain (sets layer-2 capacity) | threshold (subtracts from $Y^{(1)}_k$) |
| Layer-2 logic | $j$ gated by $k$ | coincidence of $j$ **and** $k$ |

In MLP2 the two hidden transcripts are the two halves of the output reaction: they bind each other
($a_2$, $d_2$) and the complex converts to the output ($v_2$). Neither is regenerated, so layer 2
is **not** catalytic -- both hidden nodes must be ON for any output, which makes the output node an
AND-like coincidence detector. The constitutive cassette ($\theta$) produces $Z^{(2)}$, which
sequesters $Y^{(1)}_k$ into $Q$ and so sets how much node $k$ must transcribe before it registers.

__Wiring between the layers.__ As in MLP1, the released transcripts of layer 1 *are* the input
species of layer 2 -- they are not re-transcribed from a plasmid. Writing it this way (rather than
composing two isolated activation functions) makes the downstream loading/retroactivity terms
appear automatically and avoids double counting.

__Layer 1 -- STAR perceptrons__ ($n \in \{j, k\}$, target $P^{(1)}_n$ is catalytic and conserved):
\begin{align*}
&\dot{m}^{(1)}_n = \textstyle\sum_i \alpha^{+}_{n,i} x_i - \phi m^{(1)}_n - a^{(1)}_n p^{(1)}_n m^{(1)}_n + d^{(1)}_n c^{(1)}_n - b^{(1)}_n z^{(1)}_n m^{(1)}_n + u^{(1)}_n q^{(1)}_n\\
&\dot{z}^{(1)}_n = \textstyle\sum_i \alpha^{-}_{n,i} x_i - \phi z^{(1)}_n - b^{(1)}_n z^{(1)}_n m^{(1)}_n + u^{(1)}_n q^{(1)}_n\\
&\dot{p}^{(1)}_n = d^{(1)}_n c^{(1)}_n - a^{(1)}_n p^{(1)}_n m^{(1)}_n + v^{(1)}_n c^{(1)}_n\\
&\dot{c}^{(1)}_n = a^{(1)}_n p^{(1)}_n m^{(1)}_n - d^{(1)}_n c^{(1)}_n - v^{(1)}_n c^{(1)}_n\\
&\dot{q}^{(1)}_n = b^{(1)}_n z^{(1)}_n m^{(1)}_n - u^{(1)}_n q^{(1)}_n - \phi q^{(1)}_n
\end{align*}

__Layer-1 outputs (= layer-2 inputs).__ These carry the extra terms for being consumed downstream.
Both are consumed by the $a_2$ gate reaction; only $y^{(1)}_k$ is additionally sequestered:
\begin{align*}
&\dot{y}^{(1)}_j = v^{(1)}_j c^{(1)}_j - \phi y^{(1)}_j - a_2 y^{(1)}_j y^{(1)}_k + d_2 c^{(2)}\\
&\dot{y}^{(1)}_k = v^{(1)}_k c^{(1)}_k - \phi y^{(1)}_k - a_2 y^{(1)}_j y^{(1)}_k + d_2 c^{(2)} - b_2 y^{(1)}_k z^{(2)} + u_2 q^{(2)}
\end{align*}

__Layer 2 -- coincidence node__:
\begin{align*}
&\dot{z}^{(2)} = \theta - \phi z^{(2)} - b_2 y^{(1)}_k z^{(2)} + u_2 q^{(2)}\\
&\dot{c}^{(2)} = a_2 y^{(1)}_j y^{(1)}_k - d_2 c^{(2)} - \phi c^{(2)} - v_2 c^{(2)}\\
&\dot{y}^{(2)} = v_2 c^{(2)} - \delta_2 y^{(2)}\\
&\dot{q}^{(2)} = b_2 y^{(1)}_k z^{(2)} - u_2 q^{(2)} - \phi q^{(2)}
\end{align*}

Where:
-   $X_1, X_2 = \text{Input plasmids (each carrying an } \alpha^{+} \text{ and an } \alpha^{-} \text{ cassette per hidden node)}$
-   $M^{(1)}_n = \text{STAR for hidden node } n$
-   $Z^{(1)}_n = \text{Sequestering RNA for hidden node } n$
-   $P^{(1)}_n = \text{Target RNA (paused elongation complex) of hidden node } n$
-   $C^{(1)}_n = \text{STAR-Target complex of hidden node } n$
-   $Y^{(1)}_n = \text{Released transcript of hidden node } n \text{ (= a layer-2 input)}$
-   $Q^{(1)}_n = \text{Sequestered STAR of hidden node } n$
-   $Z^{(2)} = \text{Layer-2 sequestering RNA, made constitutively at } \theta$
-   $C^{(2)} = \text{Output complex } (Y^{(1)}_j \cdot Y^{(1)}_k)$
-   $Y^{(2)} = \text{Network output}$
-   $Q^{(2)} = \text{Sequestered layer-2 input } (Y^{(1)}_k \cdot Z^{(2)})$

<u>Simulated Decision Boundaries & Activation Functions</u>

In [ ]:
#Function to help calculate the ODEs
#X --> Sets all concentrations to 0 at the start, except the layer-1 targets (conserved)
def mlpODEs(X, t, x1, x2, rates):

    #Layer 1 node j, layer 1 node k, then layer 2
    (m_j, z_j, p_j, c_j, y_j, q_j,
     m_k, z_k, p_k, c_k, y_k, q_k,
     z_2, c_2, y_2, q_2) = X

    #Unpack rates from array
    alphaPos = rates["alphaPos"]  #Transcription of STAR per unit input, [node, input]
    alphaNeg = rates["alphaNeg"]  #Transcription of sequestering RNA per unit input, [node, input]
    phi = rates["phi"]          #Decay of RNA (shared)

    a_j = rates["a_j"]          #Binding of STAR & target, node j
    d_j = rates["d_j"]          #Dissociation of STAR & target, node j
    v_j = rates["v_j"]          #Release of transcript, node j
    b_j = rates["b_j"]          #Binding of STAR & sequesterer, node j
    u_j = rates["u_j"]          #Dissociation of STAR & sequesterer, node j

    a_k = rates["a_k"]          #Same, for node k
    d_k = rates["d_k"]
    v_k = rates["v_k"]
    b_k = rates["b_k"]
    u_k = rates["u_k"]

    theta   = rates["theta"]    #Production of the layer-2 sequestering RNA Z2
    a_2     = rates["a_2"]      #Binding of the two layer-1 transcripts to each other
    d_2     = rates["d_2"]      #Dissociation
    v_2     = rates["v_2"]      #Conversion of the output complex to output
    delta_2 = rates["delta_2"]  #Decay of output RNA
    b_2     = rates["b_2"]      #Binding of y_k & Z2
    u_2     = rates["u_2"]      #Dissociation

    #Weight-summed transcription into each hidden node --> alphaPos/alphaNeg are [node, input]
    xin   = np.array([x1, x2])
    prodM = alphaPos @ xin
    prodZ = alphaNeg @ xin

    #LAYER 1, NODE j --> STAR kinetics, target is catalytic so p_j + c_j is conserved
    dm_j = prodM[0] - phi*m_j - a_j*p_j*m_j + d_j*c_j - b_j*z_j*m_j + u_j*q_j
    dz_j = prodZ[0] - phi*z_j - b_j*z_j*m_j + u_j*q_j
    dp_j = d_j*c_j - a_j*p_j*m_j + v_j*c_j
    dc_j = a_j*p_j*m_j - d_j*c_j - v_j*c_j
    dq_j = b_j*z_j*m_j - u_j*q_j - phi*q_j

    #LAYER 1, NODE k
    dm_k = prodM[1] - phi*m_k - a_k*p_k*m_k + d_k*c_k - b_k*z_k*m_k + u_k*q_k
    dz_k = prodZ[1] - phi*z_k - b_k*z_k*m_k + u_k*q_k
    dp_k = d_k*c_k - a_k*p_k*m_k + v_k*c_k
    dc_k = a_k*p_k*m_k - d_k*c_k - v_k*c_k
    dq_k = b_k*z_k*m_k - u_k*q_k - phi*q_k

    #LAYER-1 OUTPUTS == LAYER-2 INPUTS
    #The two transcripts bind EACH OTHER to make the output complex --> the a_2 term is shared
    #y_j is consumed only by that gate reaction
    dy_j = v_j*c_j - phi*y_j - a_2*y_j*y_k + d_2*c_2
    #y_k is consumed by the gate reaction AND by sequestration with the constitutive Z2
    dy_k = v_k*c_k - phi*y_k - a_2*y_j*y_k + d_2*c_2 - b_2*y_k*z_2 + u_2*q_2

    #LAYER 2 --> coincidence node, not catalytic (neither input is regenerated)
    dz_2 = theta - phi*z_2 - b_2*y_k*z_2 + u_2*q_2
    dc_2 = a_2*y_j*y_k - d_2*c_2 - phi*c_2 - v_2*c_2
    dy_2 = v_2*c_2 - delta_2*y_2
    dq_2 = b_2*y_k*z_2 - u_2*q_2 - phi*q_2

    return [dm_j, dz_j, dp_j, dc_j, dy_j, dq_j,
            dm_k, dz_k, dp_k, dc_k, dy_k, dq_k,
            dz_2, dc_2, dy_2, dq_2]


#Index of the network output y_2 in the state vector --> Used everywhere downstream
OUT = 14

Layer 1 ($n \in \{j,k\}$, STAR -- the target is catalytic so there is no $\phi$ on $c$):

$K_{C}^{(1)} = \frac{v^{(1)}_n + d^{(1)}_n}{a^{(1)}_n}\\$

$K_{Q}^{(1)} = \frac{\phi + u^{(1)}_n}{b^{(1)}_n}$

Layer 2 (coincidence node -- the output complex decays, so $\phi$ appears in $K_C$):

$K_{C}^{(2)} = \frac{\phi + v_2 + d_2}{a_2}\\$

$K_{Q}^{(2)} = \frac{\phi + u_2}{b_2}$

The layer-2 sequesterer pool is capped by its own production and decay: at steady state
$z^{(2)} + q^{(2)} \le \theta / \phi$. That ceiling, not $b_2$, is what limits how much of
$y^{(1)}_k$ can be held off, so $\theta$ sets the threshold and $\phi$ scales it.

In [ ]:
# Define a grid of input values (x1, x2)
x1 = np.linspace(0, 1, 50)
x2 = np.linspace(0, 1, 50)
X1, X2 = np.meshgrid(x1, x2)

#Activation function inputs
x1Vals = np.linspace(0, 1, 50)
x2Vals = [0.5, 0.5, 0.5]

# Simulation time --> long enough for the 16-species system to reach steady state
t = np.linspace(0, 300, 600)

#Total layer-1 target supply --> p + c is conserved at this value for each hidden node
#It is an initial condition because that is the only place a conserved quantity can live
pTot1 = 0.5

#Initial species states --> layer-1 targets start at pTot1 because they are conserved
#z_2 starts at 0 and self-fills from theta, so its initial value does not matter
X0 = [0,      #m_j
      0,      #z_j
      pTot1,  #p_j --> P is never produced in STAR --> Needs to be given
      0,      #c_j
      0,      #y_j
      0,      #q_j
      0,      #m_k
      0,      #z_k
      pTot1,  #p_k
      0,      #c_k
      0,      #y_k
      0,      #q_k
      0,      #z_2
      0,      #c_2
      0,      #y_2
      0]      #q_2

#Extra input weights --> Production terms become alphaPos @ (w1*X1, w2*X2)
w1 = 1
w2 = 1

#Kinetics parameters --> theta is varied across variants (the layer-2 threshold)
#alphaPos/alphaNeg are stacked per variant --> each entry is a [node, input] matrix

                      #X1   X2
alphaPos1 = np.array([[1.0, 0.6],    #node j
                      [0.6, 1.0]])   #node k
alphaNeg1 = np.array([[0.2, 0.4],    #node j
                      [0.4, 0.2]])   #node k

ratesDefault = {
    'alphaPos': np.array([alphaPos1, alphaPos1, alphaPos1]),
    'alphaNeg': np.array([alphaNeg1, alphaNeg1, alphaNeg1]),
    'phi':     np.array([0.1, 0.1, 0.1]),
    'a_j':     np.array([10, 10, 10]),
    'd_j':     np.array([1, 1, 1]),
    'v_j':     np.array([1, 1, 1]),
    'b_j':     np.array([200, 200, 200]),
    'u_j':     np.array([0.2, 0.2, 0.2]),
    'a_k':     np.array([10, 10, 10]),
    'd_k':     np.array([1, 1, 1]),
    'v_k':     np.array([1, 1, 1]),
    'b_k':     np.array([200, 200, 200]),
    'u_k':     np.array([0.2, 0.2, 0.2]),
    'theta':   np.array([0.1, 0.3, 1.0]),   #<-- the varied rate: layer-2 threshold
    'a_2':     np.array([10, 10, 10]),
    'd_2':     np.array([1, 1, 1]),
    'v_2':     np.array([1, 1, 1]),
    'delta_2': np.array([0.1, 0.1, 0.1]),
    'b_2':     np.array([200, 200, 200]),
    'u_2':     np.array([0.2, 0.2, 0.2]),
}

#getVariantRates() --> Extracts variant-specific rates from ratesDefault
def getVariantRates(ratesArray, variantIndex):
    rates = {}
    for key, arr in ratesArray.items():
        rates[key] = arr[variantIndex]
    return rates

#getVariantLabel() --> Legend label (labelBy = "alpha", "xi2", "K_C2", "K_Q2", "theta", or "None" for no legend)
def getVariantLabel(rates, labelBy):
    if labelBy in ("None", None):
        return None
    if labelBy == "alpha":
        #The two matrices side by side are too wide for a legend --> show the net drive they produce
        netDrive = rates["alphaPos"] - rates["alphaNeg"]
        return "W = " + np.array2string(netDrive, precision=2, separator=" ").replace("\n", "")
    if labelBy == "xi2":  # xi = K_Q2 / K_C2 = a_2(phi + u_2) / (b_2(phi + v_2 + d_2))
        return (f"ξ2 = {rates['a_2'] * (rates['phi'] + rates['u_2']) / (rates['b_2'] * (rates['phi'] + rates['v_2'] + rates['d_2'])):.3g}")
    if labelBy == "K_C2":
        return f"K_C2 = {(rates['phi'] + rates['v_2'] + rates['d_2']) / rates['a_2']:.3g}"
    if labelBy == "K_Q2":
        return f"K_Q2 = {(rates['phi'] + rates['u_2']) / rates['b_2']:.3g}"
    if labelBy == "zTot2":  #Ceiling on the layer-2 sequesterer pool --> the actual threshold scale
        return f"z2_tot = {rates['theta'] / rates['phi']:.3g}"
    return f"theta = {rates['theta']:.3g}"

numVariants = len(ratesDefault['phi']) #phi is in all models --> Best choice

#Bokeh color palette for heatmaps + activation functions
colors = bokeh.palettes.OrRd3

In [ ]:
# Function to do the 2D projection of decision boundaries with normalization  <-- From the EBCP LATAM 2025 Workshop
def contourf(p, x, y, z, title=None, palette="Spectral11", normal = True):
    """Make a filled contour plot given x, y, z data given in 2D arrays."""

    # Normalize the z values
    if normal:
      z_min = z.min()
      z_max = z.max()
      z_normalized = (z - z_min) / (z_max - z_min)  # Normalize to range [0, 1]
    else:
      z_normalized = z

    # Add zero padding at the boundaries for better visualization
    N = z_normalized.shape[1]
    z0 = np.c_[z_normalized, np.zeros(N)]
    z0[-1, -1] = 1.  # Ensure the padding contains a value within range

    # Plot the normalized values
    p.image(
        image=[z0],
        x=x.min(),
        y=y.min(),
        dw=(x.max() - x.min()) * (1 + 1 / N),
        dh=x.max() - x.min(),
        palette=palette,
        alpha=0.8,
    )

    # Color mapping based on the normalized z0 values
    color = LinearColorMapper(palette=palette, low=z0.min(), high=z0.max())
    cb = ColorBar(color_mapper=color, location=(0, 0), width=10)
    p.add_layout(cb, 'right')

    return ()

In [ ]:
#Array to store all results
results = []

#Range through each variant --> Solve ODEs + Plot heatmap + Plot activation function --> Append to Results
#NOTE: 50x50 grid x 3 variants x a 16-species system --> this cell takes a few minutes
for variantIndex in range(numVariants):
    #Get rates for this current iteration's variant
    rates = getVariantRates(ratesDefault, variantIndex)

    #ODE SOLUTIONS
    #Array for storing scipy.integrate solutions
    gridSolutions = [[None for _ in range(len(x1))] for _ in range(len(x2))]

    # Initialize a matrix to store the steady-state output y
    Y = np.zeros_like(X1)

    # Iterate over the grid values for all variable inputs X1 and X2 --> Solve the ODEs
    for i in range(len(x1)):
        for j in range(len(x2)):
            solutions = scipy.integrate.odeint(mlpODEs, X0, t, args=(w1 * X1[j, i], w2 * X2[j, i], rates))
            gridSolutions[j][i] = solutions
            Y[j, i] = solutions[-1, OUT] #Extract the steady state output y_2

    #HEATMAPS
    p_heatmap = figure(width=350, height=300, title=f"Variant {variantIndex}")

    #Make the heatmap
    contourf(
        p_heatmap,
        X1,
        X2,
        Y,
        palette=bokeh.palettes.Oranges8[::-1],
        normal=True
    )

    p_heatmap.xaxis.axis_label = "X1"
    p_heatmap.yaxis.axis_label = "X2"

    #ACTIVATION FUNCTIONS
    activationFunctions = {}
    p_activation = figure(width=350, height=300, title=f"Variant {variantIndex}")

    for x2j, color in zip(x2Vals, colors):

        curves = []
        output_y = np.zeros_like(x1Vals)

        for i, x1i in enumerate(x1Vals):

            sol = scipy.integrate.odeint(
                mlpODEs, X0, t,
                args=(w1 * x1i, w2 * x2j, rates)
            )

            curves.append(sol)
            output_y[i] = sol[-1, OUT]

        activationFunctions[x2j] = curves

        p_activation.line(
            x1Vals,
            output_y,
            line_width=3,
            color=color,
            legend_label=f"X2 = {x2j}"
        )

    p_activation.xaxis.axis_label = "X1 (Input)"
    p_activation.yaxis.axis_label = "y_2 (Network Output)"
    p_activation.legend.location = "top_left"

    #Append ODE solutions, heatmaps, and activation functions for all variants into the results array
    results.append({
        "variantIndex": variantIndex,
        "rates": rates,
        "solutions": {
            "grid": gridSolutions,
            "activation": activationFunctions
        },
        "heatmaps": p_heatmap,
        "activationFunctions": p_activation
    })

In [ ]:
#Plot results
for res in results:
    print(f"\nVariant {res['variantIndex']}")
    for k, v in res["rates"].items():
        print(f"{k}: {v}")

    show(row(res["heatmaps"], res["activationFunctions"]))

In [ ]:
###Sanity checks --> run after the variant loop. Every number here is an invariant of the
###model, not a fitted result, so a failure means a term was mistyped rather than mis-tuned.
checkRates = getVariantRates(ratesDefault, 1)
sol = scipy.integrate.odeint(mlpODEs, X0, t, args=(w1 * 1.0, w2 * 1.0, checkRates))
final = sol[-1]

#1. Steady state --> the grid results are read off the last time point, so it must be converged
deriv = np.array(mlpODEs(final, t[-1], w1 * 1.0, w2 * 1.0, checkRates))
print(f"1. max |dX/dt| at t = {t[-1]:g}:  {np.abs(deriv).max():.3g}   (should be ~0)")

#2. Layer-1 target conservation --> p + c has no production or decay term, so it is pinned to pTot1
#   This catches an accidental phi or theta on the layer-1 target
print(f"2. node j  p + c = {final[2] + final[3]:.6f}   (pTot1 = {pTot1})")
print(f"   node k  p + c = {final[8] + final[9]:.6f}   (pTot1 = {pTot1})")

#3. Layer-2 sequesterer budget --> z_2 is made at theta and decays at phi, and every molecule of it
#   is either free or in q_2, so the pool is capped at theta/phi
zTot2 = final[12] + final[15]
print(f"3. z_2 + q_2 = {zTot2:.6f}   (ceiling theta/phi = {checkRates['theta'] / checkRates['phi']:.6f})")

#4. The output node is a coincidence detector --> killing EITHER hidden node must kill the output.
#   This is the structural difference from MLP1, where only node j fed the gate directly.
for killed, X0killed in (("j", list(X0)), ("k", list(X0))):
    X0killed[2 if killed == "j" else 8] = 0.0   #No target supply --> that node can never release y
    yKilled = scipy.integrate.odeint(mlpODEs, X0killed, t, args=(w1 * 1.0, w2 * 1.0, checkRates))[-1, OUT]
    print(f"4. node {killed} disabled (p_{killed}(0) = 0):  y_2 = {yKilled:.6f}   (should be 0)")

#5. theta is a threshold, not a gain --> raising it can only lower the output, and it can never
#   lift the output off zero at the origin (so it is not a bias)
print("5. theta sweep at (x1, x2) = (1, 1), and at the origin:")
for th in (0.0, 0.1, 0.3, 1.0, 3.0):
    r = dict(checkRates, theta=th)
    yOn  = scipy.integrate.odeint(mlpODEs, X0, t, args=(w1 * 1.0, w2 * 1.0, r))[-1, OUT]
    yOff = scipy.integrate.odeint(mlpODEs, X0, t, args=(0.0, 0.0, r))[-1, OUT]
    print(f"   theta = {th:<4g}  y_2(1,1) = {yOn:.4f}   y_2(0,0) = {yOff:.4f}")

In [ ]:
### plotActivationFunctionVariants() --> Plots the network activation function for every rate variant on the same figure, holding X2 fixed, so the effect of the varied rate(s) is directly comparable
def plotActivationFunctionVariants(ratesDefault, x2_fixed, x1Vals, w1, w2, X0, t, labelBy="None"):
    numVariants = len(ratesDefault['phi'])  # phi is in all models --> Best choice
    colors = bokeh.palettes.OrRd3  # Reuse the same palette as the rest of the notebook

    p_activation = figure(width=500, height=400, x_range=(0, 1), title=f"Network Activation Function Across Rate Variants")

    for variantIndex in range(numVariants):
        rates = getVariantRates(ratesDefault, variantIndex)

        output_y = np.zeros_like(x1Vals)
        for i, x1i in enumerate(x1Vals):
            sol = scipy.integrate.odeint(
                mlpODEs, X0, t,
                args=(w1 * x1i, w2 * x2_fixed, rates)
            )
            output_y[i] = sol[-1, OUT]  # Steady-state output y_2

        line_kwargs = {"line_width": 3, "color": colors[variantIndex]}
        label = getVariantLabel(rates, labelBy)
        if label is not None:
            line_kwargs["legend_label"] = label  #labelBy="None" --> No legend

        p_activation.line(x1Vals, output_y, **line_kwargs)

    p_activation.xaxis.axis_label = "X1 (Input)"
    p_activation.yaxis.axis_label = "y_2 (Network Output)"
    if p_activation.legend:
        p_activation.legend.location = "top_left"

    show(p_activation)
    return p_activation

In [ ]:
### Example call --> Hold X2 fixed at one of the existing x2Vals
p1 = plotActivationFunctionVariants(ratesDefault, x2_fixed=x2Vals[0], x1Vals=x1Vals, w1=w1, w2=w2, X0=X0, t=t, labelBy="theta")

#Save plot
#p1.output_backend = "svg"
#export_svgs(p1, filename="Simulation_Plots/mlp2/mlp2_activation_variants.svg")

In [ ]:
###plotDecisionBoundaryVariants() --> Extracts the Y = threshold contour (the decision boundary) for each rate variant and overlays them on one (X1, X2) plot
def plotDecisionBoundaryVariants(ratesDefault, threshold, x1, x2, X1, X2, w1, w2, X0, t, relative=False, labelBy="None", colors=None):
    numVariants = len(ratesDefault['phi'])  # phi is in all models --> Best choice
    if colors is None:
        colors = bokeh.palettes.OrRd3 if numVariants <= 3 else bokeh.palettes.Category10[10]  #OrRd3 only has 3 colors --> Use Category10 for more variants

    p_boundary = figure(width=500, height=400, x_range=(0, 1), y_range=(0, 1), title=f"Decision Boundary Across Rate Variants (Y = {threshold}{' * max(Y)' if relative==True else ''})")

    for variantIndex in range(numVariants):
        rates = getVariantRates(ratesDefault, variantIndex)

        # Solve the ODEs over the full (X1, X2) grid to get steady-state output Y
        Y = np.zeros_like(X1)
        Yj = np.zeros_like(X1)  # Steady-state output of hidden node j
        Yk = np.zeros_like(X1)  # Steady-state output of hidden node k
        for i in range(len(x1)):
            for j in range(len(x2)):
                sol = scipy.integrate.odeint(mlpODEs, X0, t, args=(w1 * X1[j, i], w2 * X2[j, i], rates))
                Y[j, i]  = sol[-1, OUT]  # Steady-state output y_2
                Yj[j, i] = sol[-1, 4]    # Steady-state free y_j
                Yk[j, i] = sol[-1, 10]   # Steady-state free y_k

        # relative=True --> threshold is a fraction of this variant's max Y
        level = threshold * Y.max() if relative else threshold

        # Extract the Y = level contour --> This is the decision boundary
        contours = measure.find_contours(Y, level=level)

        if len(contours) == 0:
            print(f"Variant {variantIndex}: Y never crosses {level:.3g} on this grid (max Y = {Y.max():.3g}) --> no boundary to plot")
            continue

        #A contour of only a few points is a degenerate blob (usually a threshold set just above
        #Y = 0 at the origin) --> it draws as nothing, so say so rather than failing silently
        longest = max(len(c) for c in contours)
        if longest < 5:
            print(f"Variant {variantIndex}: boundary at {level:.3g} is degenerate ({longest} points) --> raise the threshold")

        for c, contour in enumerate(contours):
            #find_contours returns [row_idx, col_idx] into the Y grid --> map indices back to real x1, x2 values
            x1_contour = np.interp(contour[:, 1], np.arange(len(x1)), x1)
            x2_contour = np.interp(contour[:, 0], np.arange(len(x2)), x2)

            line_kwargs = {"line_width": 3, "color": colors[variantIndex]}
            if c == 0 and labelBy not in ("None", None):
                line_kwargs["legend_label"] = getVariantLabel(rates, labelBy)  #Only label once per variant

            p_boundary.line(x1_contour, x2_contour, **line_kwargs)

    p_boundary.xaxis.axis_label = "X1 (Input plasmid 1)"
    p_boundary.yaxis.axis_label = "X2 (Input plasmid 2)"
    if p_boundary.legend:
        p_boundary.legend.location = "top_left"

    show(p_boundary)
    return p_boundary

In [ ]:
###threshold = Y value (fraction of max Y when relative=True) --> HELD CONSTANT
###Use relative=True with 0.3-0.5: an absolute threshold near 0 picks out only the origin cell
p2 = plotDecisionBoundaryVariants(ratesDefault, threshold=0.5, x1=x1, x2=x2, X1=X1, X2=X2, w1=w1, w2=w2, X0=X0, t=t, relative=True, labelBy="theta")

#Save plot
#p2.output_backend = "svg"
#export_svgs(p2, filename="Simulation_Plots/mlp2/mlp2_decision_boundary_variants.svg")

In [ ]:
###Alpha comparison --> Variant 0 = baseline, Variants 1-2 scale alphaPos, Variants 3-4 scale alphaNeg
#Kinetics are held at the Variant 0 values so only the weight matrices differ
nAlpha = 5
ratesAlpha = {
    'alphaPos': np.array([alphaPos1, 0.5 * alphaPos1, 0.25 * alphaPos1, alphaPos1, alphaPos1]),
    'alphaNeg': np.array([alphaNeg1, alphaNeg1, alphaNeg1, 0.25 * alphaNeg1, 0.5 * alphaNeg1]),
    'phi':     np.full(nAlpha, 0.1),
    'a_j':     np.full(nAlpha, 10),
    'd_j':     np.full(nAlpha, 1),
    'v_j':     np.full(nAlpha, 1),
    'b_j':     np.full(nAlpha, 200),
    'u_j':     np.full(nAlpha, 0.2),
    'a_k':     np.full(nAlpha, 10),
    'd_k':     np.full(nAlpha, 1),
    'v_k':     np.full(nAlpha, 1),
    'b_k':     np.full(nAlpha, 200),
    'u_k':     np.full(nAlpha, 0.2),
    'theta':   np.full(nAlpha, 0.3),
    'a_2':     np.full(nAlpha, 10),
    'd_2':     np.full(nAlpha, 1),
    'v_2':     np.full(nAlpha, 1),
    'delta_2': np.full(nAlpha, 0.1),
    'b_2':     np.full(nAlpha, 200),
    'u_2':     np.full(nAlpha, 0.2),
}

#Colors --> Baseline black, alphaPos changes in reds, alphaNeg changes in blues
alphaColors = ["black", "#fc9272", "#de2d26", "#9ecae1", "#3182bd"]

p3 = plotDecisionBoundaryVariants(ratesAlpha, threshold=0.5, x1=x1, x2=x2, X1=X1, X2=X2, w1=w1, w2=w2, X0=X0, t=t, relative=True, labelBy="alpha", colors=alphaColors)

#Save plot
#p3.output_backend = "svg"
#export_svgs(p3, filename="Simulation_Plots/mlp2/mlp2_alpha_decision_boundary.svg")

In [ ]:
###plotHeatmap() --> Standalone heatmap of the steady-state output Y over the (X1, X2) grid for a single rate variant
def plotHeatmap(ratesDefault, variantIndex, x1, x2, X1, X2, w1, w2, X0, t, normal=True, labelBy="None"):
    rates = getVariantRates(ratesDefault, variantIndex)

    # Solve the ODEs over the full (X1, X2) grid to get steady-state output Y
    Y = np.zeros_like(X1)
    for i in range(len(x1)):
        for j in range(len(x2)):
            sol = scipy.integrate.odeint(mlpODEs, X0, t, args=(w1 * X1[j, i], w2 * X2[j, i], rates))
            Y[j, i] = sol[-1, OUT]  # Steady-state output y_2

    # Title --> Variant label (alpha, xi2, K_C2, K_Q2, theta, zTot2) if given, otherwise the variant index
    label = getVariantLabel(rates, labelBy)
    title = label if label is not None else f"Heatmap"

    p_heatmap = figure(width=450, height=400, x_range=(0, 1), y_range=(0, 1), title=title)

    #Make the heatmap --> normal=True normalizes Y to [0, 1]
    contourf(
        p_heatmap,
        X1,
        X2,
        Y,
        palette=bokeh.palettes.Oranges8[::-1],
        normal=normal
    )

    p_heatmap.xaxis.axis_label = "X1 (Input plasmid 1)"
    p_heatmap.yaxis.axis_label = "X2 (Input plasmid 2)"

    show(p_heatmap)
    return p_heatmap

def heatmapForIllustrator(filename):
    with open(filename) as f:
        svg = f.read()
    # Illustrator only reads xlink:href for embedded images
    svg = re.sub(r'(<image[^>]*?)\shref=', r'\1 xlink:href=', svg)
    if 'xmlns:xlink' not in svg:
        svg = svg.replace('<svg ', '<svg xmlns:xlink="http://www.w3.org/1999/xlink" ', 1)
    with open(filename, "w") as f:
        f.write(svg)

In [ ]:
p4 = plotHeatmap(ratesDefault, variantIndex=0, x1=x1, x2=x2, X1=X1, X2=X2, w1=w1, w2=w2, X0=X0, t=t, normal=True, labelBy="None")
p4.output_backend = "svg"
for fn in export_svgs(p4, filename="Simulation_Plots/mlp2/mlp2_heatmap1.svg"):
    heatmapForIllustrator(fn)

In [ ]:
p5 = plotHeatmap(ratesDefault, variantIndex=1, x1=x1, x2=x2, X1=X1, X2=X2, w1=w1, w2=w2, X0=X0, t=t, normal=True, labelBy="None")
p5.output_backend = "svg"
for fn in export_svgs(p5, filename="Simulation_Plots/mlp2/mlp2_heatmap2.svg"):
    heatmapForIllustrator(fn)

In [ ]:
p6 = plotHeatmap(ratesDefault, variantIndex=2, x1=x1, x2=x2, X1=X1, X2=X2, w1=w1, w2=w2, X0=X0, t=t, normal=True, labelBy="None")
p6.output_backend = "svg"
for fn in export_svgs(p6, filename="Simulation_Plots/mlp2/mlp2_heatmap3.svg"):
    heatmapForIllustrator(fn)

In [ ]:
#3D-Plot of the network activation function
fig = go.Figure(data=[
    go.Surface(z=Y, x=X1, y=X2, colorscale="Viridis")
])

fig.update_layout(
    scene=dict(
        xaxis_title='Input plasmid 1 (X1)',
        yaxis_title='Input plasmid 2 (X2)',
        zaxis_title='Network Output (Y2)'
    ),
    width=800,
    height=600
)

fig.show()

<u>Separability of the Network as Drawn</u>

Every production term that depends on an input is strictly proportional to an input plasmid, so
each hidden node's net drive carries no constant term:

$$\text{net}_n(x) = \sum_i \left(\alpha^{+}_{n,i} - \alpha^{-}_{n,i}\right) x_i$$

This means $\text{net}_n(0,0) = 0$ for both hidden nodes, so **every hidden decision boundary passes
through the origin**. The hidden layer can rotate its two boundaries freely but cannot translate
them off $(0,0)$.

$\theta$ does not rescue this. It is input-independent, so it looks like a candidate bias, but it
produces the *sequesterer* $Z^{(2)}$, which only ever subtracts from $y^{(1)}_k$. A negative bias
cannot lift $y^{(2)}(0,0)$ off zero -- and in fact nothing can here, because
$\dot{c}^{(2)} \propto y^{(1)}_j y^{(1)}_k$ and both hidden outputs are zero at the origin. The
output is pinned to $0$ at $(0,0)$ by the *product*, more firmly than in MLP1.

The cell below measures this directly. It reports the **margin**
$\min(\text{ON}) - \max(\text{OFF})$ for each target function -- positive means the function is
realized, and the margin should be a decent fraction of the output range to be usable. The search
varies $\theta$ alongside the weights, since $\theta$ is this architecture's extra free knob.

In [ ]:
###Separability check --> the network exactly as drawn, no constitutive cassettes feeding the gate
ratesCheck = getVariantRates(ratesDefault, 1)


#truthTable() --> Steady-state output at the four corners of the input square
def truthTable(rates):
    return {(xa, xb): scipy.integrate.odeint(mlpODEs, X0, t, args=(w1 * xa, w2 * xb, rates))[-1, OUT]
            for xa in (0.0, 1.0) for xb in (0.0, 1.0)}


#margin() --> min(ON) - max(OFF). Positive means the target function is realized.
def margin(T, onCorners):
    on  = [T[c] for c in onCorners]
    off = [T[c] for c in T if c not in onCorners]
    return min(on) - max(off)


#withAlphas() --> Copy of a rates dict with the weight matrices (and optionally theta) swapped out
def withAlphas(rates, aPos, aNeg, theta=None):
    swapped = dict(rates)
    swapped["alphaPos"] = aPos
    swapped["alphaNeg"] = aNeg
    if theta is not None:
        swapped["theta"] = theta
    return swapped


TARGETS = {
    "XOR": [(0.0, 1.0), (1.0, 0.0)],
    "OR":  [(0.0, 1.0), (1.0, 0.0), (1.0, 1.0)],
    "AND": [(1.0, 1.0)],
}

### The weights used throughout this notebook
T0 = truthTable(ratesCheck)
print("Network as configured above:")
print("  X1   X2  |   y_2")
for (xa, xb), y in sorted(T0.items()):
    print(f"  {xa:.1f}  {xb:.1f}  |  {y:.4f}")

outputRange = max(T0.values()) - min(T0.values())
netDrive = ratesCheck["alphaPos"] - ratesCheck["alphaNeg"]
print()
for n, name in enumerate(("j", "k")):
    print(f"  net drive of node {name}:  {netDrive[n, 0]:+.2f} * X1  {netDrive[n, 1]:+.2f} * X2"
          f"   --> 0 at the origin")

### Can ANY weights reach each target? --> random search over the alpha matrices and theta
#NOTE: a few hundred ODE solves --> this takes roughly half a minute
rng = np.random.default_rng(0)
nTrials = {"XOR": 120, "OR": 50, "AND": 50}

#A margin must be judged on an ABSOLUTE scale. Scoring it as a fraction of the winning trial's
#own output range would call a dead all-zero network (margin 0 on range 0) a success, so the bar
#is a fixed fraction of the configured network's corner range instead.
scale = outputRange
print(f"\nBest margin found over random weight sets (bar = 10% of the configured range, {scale:.3g}):")
print("NOTE: alphas up to 3 make the system stiff --> an occasional ODEintWarning is expected here.")
for name, onCorners in TARGETS.items():
    best = -np.inf
    bestRange = 0.0
    for _ in range(nTrials[name]):
        trialRates = withAlphas(ratesCheck, rng.uniform(0, 3, (2, 2)), rng.uniform(0, 3, (2, 2)),
                                theta=rng.uniform(0, 2))
        T = truthTable(trialRates)
        m = margin(T, onCorners)
        if m > best:
            best, bestRange = m, max(T.values()) - min(T.values())
    verdict = "reachable" if best > 0.1 * scale else "NOT reachable"
    #Reporting the winner's own range too --> a margin of 0 on a range of 0 is a dead network,
    #not a solution, and that is only visible when both numbers are shown
    print(f"  {name:<4} ({nTrials[name]:>3} trials):  margin {best:+.4f}  on its own range {bestRange:.4f}   --> {verdict}")

print("\nXOR fails because both hidden boundaries are pinned to the origin, and the output node")
print("multiplies the two hidden outputs, so y_2(0,0) = 0 no matter how theta is set.")
print("theta is a threshold on node k, not a bias --> it can only subtract.")